In [1]:
%load_ext line_profiler

In [2]:
from rabtracks import *

MIN_OVERLAP = 3
MAX_DIST = 400

In [3]:
def load_ibtracs(b):
    filepath = "ibtracs/ibtracs_"+b+".pkl"
    with open(filepath, "rb") as f:
        ib = pkl.load(f)
    return ib

In [4]:
def load_tracks():
    # Find available files
    TRACK_flist = glob("RA_tracks/TRACK-*.pkl") 
    SyCLoPS_flist = glob("RA_tracks/SyCLoPS-*.pkl")
    # Load them
    tracks = {}
    for f in tqdm(TRACK_flist + SyCLoPS_flist):
        name = f.split("/")[-1].split('.')[0]
        with open(f, "rb") as file:
            tracks[name] = pkl.load(file)
    return tracks

In [5]:
def matching_step1(tracks_source, source_name, ib, MIN_OVERLAP=MIN_OVERLAP, MAX_DIST=MAX_DIST):
    """Step 1: For each source, perform individual matching and treat duplicates"""
    
    ## Step 1a: Match the source with ibtracs
    M = huracanpy.assess.match([ib, tracks_source], ["ib", source_name], min_overlap = MIN_OVERLAP, max_dist = MAX_DIST, 
                              tracks1_is_ref = True,   # Treat the duplicates where one reanalysis tracks has several corresponding IBTrACS:
                                                       # Keep the couple with longest overlap
                              )
    ## Step 1b: Treat the duplicates where several RA tracks correspond to 1 obs.: Combined them together
    ### Merge with M to subset matched trackss and assign matching distance
    tracks_source = tracks_source.to_dataframe().merge(M.rename(columns = {"id_"+source_name:"track_id"})[["track_id", "dist"]])
    ### Update merged ids
    new_ids = new_ids = M["id_"+source_name].astype(str).groupby(M.id_ib).apply(lambda s: '+'.join(s))
    replace = M.join(new_ids, on = "id_ib", lsuffix = "_old", rsuffix = "_new")[["id_"+source_name+"_old", "id_"+source_name+"_new"]]
    tracks_source["track_id"] = tracks_source["track_id"].replace(replace.set_index("id_"+source_name+"_old")["id_"+source_name+"_new"].to_dict())
    ### When several times are for one track_id: keep the one belonging to the closest tracks to IBTrACS
    tracks_source = tracks_source.sort_values("dist").groupby(["track_id", "time"]).first().reset_index()
    ### Retransform into xarray
    tracks_source = tracks_source.to_xarray().rename({"index":"record"})
    return tracks_source

In [6]:
def matching(tracks, ib):
    # Step 1: For each source, perform individual matching and treat duplicates
    tracks_combined = {}
    for ds in tqdm(tracks): 
        tracks_combined[ds] = matching_step1(tracks[ds].copy(), ds, ib)

    # Step 2: Redo the matching now that the tracks have been merged
    M = huracanpy.assess.match([ib, ib, *tracks_combined.values()], ["ib", "ib2", *tracks_combined.keys()], 
                               min_overlap = MIN_OVERLAP, max_dist = MAX_DIST, 
                               tracks1_is_ref = True,
                              ).drop(columns = "id_ib2").drop_duplicates()
    # Note : The double ib is a trick to make sure all tracks from IBTrACS are included, since they all match with themselves
    # Remove matches where IBTrACS is not involved: IBTrACS is considered ground truth for whether a track was tropical
    M  = M[~(M.id_ib.isna())]
    
    return M, tracks_combined

In [7]:
def gathering(tracks_combined, M, ib):
    # Step 3: Gather tracks together
    ## Gather all the RA tracks that matched with a given IBTrACS track
    tracks_matched = []
    for id_ib in tqdm(np.unique(ib.track_id)): # Loop over rows of M 
        track = ib[["track_id", "lon", "lat", "time",]].hrcn.sel_id(id_ib).set_coords("time").swap_dims({"record":"time"}
                                                        ).assign_coords(source = "IBTrACS").expand_dims(dim="source")
        if len(track.time) > 1:
            # Matches
            m = M[M.id_ib == id_ib]
            
            # Storm attributes
            name = ib.hrcn.sel_id(id_ib).name[0].values
        
            # Merge all data about the track in one dataset
            if len(m)>0:
                for source in tracks_combined:
                    source_id = getattr(m, "id_"+source).values[0] # ID in the given source
                    vars2keep = [v for v in ["track_id", "lon", "lat", "pres", "wind10", "label","vo850", "is_tc", "lpsarea", 'lon_pres', 'lat_pres', 'lon_wind', 'lat_wind',] if v in list(tracks_combined[source].variables.keys())]
                    tid = tracks_combined[source].hrcn.sel_id(source_id).set_coords("time").swap_dims({"record":"time"}
                                                                                    ).reset_coords()[vars2keep
                            ].assign(source = source).set_coords("source")
                    track = xr.concat([track, tid], dim = "source")
            
            # Save into tracks
            tracks_matched.append(track.swap_dims({"time":"record"}).rename({"track_id":"track_id_source"}).assign(track_id = id_ib, name = name))
        
    ## Concatenate all tracks in a new dataset
    eib = xr.concat(tracks_matched, dim = "record")
    
    # Remove track_id_source
    eib = eib.drop_vars("track_id_source")
    # Filter 6-hourly
    eib = clean(eib.where(eib.time.dt.hour % 6 == 0,))

    # Add units
    eib["wind10"].attrs["units"] = "ms-1"
    eib["pres"].attrs["units"] = "hPa"
    eib["vo850"].attrs["units"] = "1e-5 s-1"
    varlist = list(eib.variables.keys())
    for v in varlist:
        if v.startswith("lon") or v.startswith("lat"):
            eib[v].attrs["units"] = "deg"
    return eib

In [8]:
def retrieve_ib_attrs(eib, ib):
    ## Transform into dataframes
    ib_df = ib.to_dataframe()
    teib_df = eib.sel(source = "IBTrACS").squeeze().reset_coords().to_dataframe()
    # Merge data
    M = teib_df[["time","track_id"]].reset_index().merge(ib_df, how = "left", on = ["time", "track_id"])
    assert len(M) == len(teib_df) # Check that no duplicate were created
    # Transform to xarray and select attributes
    M_xr = M.set_index("record").to_xarray()
    # Merge back into the RABTracks
    eib = eib.reset_coords().merge(M_xr.drop_vars(["lon", "lat",]))
    for v in list(eib.variables.keys()):
        if v in list(ibtracs_units.keys()):
            if ibtracs_units[v] is not None:
                eib[v].attrs["units"] = ibtracs_units[v]
    return eib

In [9]:
def add_translation_speed_azimuth(eib):
    S = []
    S_roll = []
    A = []
    A_roll = []
    for source in tqdm(eib.source):
        eib_source = eib.sel(source = source) # Subset source
        # Compute lon & lat rolling mean
        eib_source_df = eib_source.to_dataframe()
        lonlat_roll = eib_source_df.reset_index().set_index("record").sort_values("time").groupby(["track_id"]).rolling(5, center = True)[["lon", "lat"]].mean()
        eib_source[["lon_roll", "lat_roll",]] = lonlat_roll.reset_index().set_index(["record"]).to_xarray()[["lon", "lat"]]
        
        eib_source = clean(eib_source.where(~np.isnan(eib_source.lon),)) # Remove points where given source does not have data
        speed = huracanpy.calc.translation_speed(eib_source.lon, eib_source.lat, eib_source.time, eib_source.track_id)
        speed_roll = huracanpy.calc.translation_speed(eib_source.lon_roll, eib_source.lat_roll, eib_source.time, eib_source.track_id)
        S.append(speed)
        S_roll.append(speed_roll)
        azimuth = huracanpy.calc.azimuth(eib_source.lon, eib_source.lat, eib_source.track_id)
        azimuth_roll = huracanpy.calc.azimuth(eib_source.lon_roll, eib_source.lat_roll, eib_source.track_id)
        A.append(azimuth)
        A_roll.append(azimuth_roll)
    eib["translation_speed"] = xr.concat(S, dim = "source")
    eib["translation_speed_roll"] = xr.concat(S_roll, dim = "source")
    eib["azimuth"] = xr.concat(A, dim = "source")
    eib["azimuth_roll"] = xr.concat(A_roll, dim = "source")

    # Add units
    varlist = list(eib.variables.keys())
    for v in varlist:
        if v.startswith("translation_speed") :
            eib[v].attrs["units"] = "ms-1"
        elif v.startswith("azimuth"):
            eib[v].attrs["units"] = "deg"
    return eib

In [10]:
def add_radius_of_maximum_wind(eib):
    # Compute RMW
    RMW = huracanpy.calc.distance(
        eib.lon_pres, eib.lat_pres, eib.lon_wind, eib.lat_wind
    ) / 1e3
    RMW.attrs["units"] = "km"
    eib["radius_max_wind"] = RMW
    # Compute rolling mean
    RMW_df = eib[["time", "radius_max_wind", "track_id"]].to_dataframe()
    RMW_roll = RMW_df.reset_index().set_index("record").sort_values("time").groupby(["source", "track_id"]).rolling(5, center = True)["radius_max_wind"].mean()
    RMW_roll = RMW_roll.reset_index().set_index(["source", "record"])["radius_max_wind"].to_xarray()
    RMW_roll.attrs["units"] = "km"
    eib["radius_max_wind_roll"] = RMW_roll
    return eib

In [11]:
def load_CPS_tracks():
    # Find available files
    CPS_files = glob("RA_tracks/TRACK_CPS-*.pkl") 
    # Load them
    tracks_with_CPS = {}
    for f in CPS_files:
        name = "TRACK-"+f.split("/")[-1].split('.')[0].split("-")[-1]
        with open(f, "rb") as file:
            tracks_with_CPS[name] = pkl.load(file)
    return tracks_with_CPS

In [12]:
def add_CPS_params(eib):
    
    tracks_with_CPS = load_CPS_tracks()
        
    L = []
    for source in tqdm(tracks_with_CPS):
        # Transform the sources into dataframes
        t_CPS_df = tracks_with_CPS[source].to_dataframe()
        teib_df = eib.sel(source = source).squeeze().reset_coords().to_dataframe()
        for var in ["lon", "lat"]: # Round up the coordinates for the matching to work
            N = 0
            t_CPS_df[var] = np.round(t_CPS_df[var], N)
            teib_df[var] = np.round(teib_df[var], N)
        # Merge RABTracks and CPS source on lon, lat and time
        t_CPS_df = t_CPS_df[~t_CPS_df[["lon", "lat", "time",]].duplicated()] # Remove duplicates
        M = teib_df[["lon", "lat", "time",]].reset_index().merge(t_CPS_df[["lon", "lat", "time", "vtl", "vtu", "b"]], how = "left", on = ["lon", "lat", "time"])
        assert len(M) == len(teib_df) # Check that no duplicate were created
        M_xr = M.set_index("record").to_xarray().assign(source = source).set_coords("source")[["vtu", "vtl", "b"]] # convert back to source and format
        L.append(M_xr)
    CPS = xr.concat(L, dim = "source") # Merge the CPS variables over sources
    eib = eib.merge(CPS) # Merge back into the full RABTracks source

    # Compute 1-day rolling means of CPS parameters
    eib_df = eib.to_dataframe()
    CPS_roll = eib_df.reset_index().set_index("record").sort_values("time").groupby(["source", "track_id"]).rolling(5, center = True)[["vtu", "vtl", "b"]].mean()
    eib[["vtu_roll", "vtl_roll", "b_roll"]] = CPS_roll.reset_index().set_index(["source", "record"]).to_xarray()[["vtu", "vtl", "b"]]
    
    # Add units
    varlist = list(eib.variables.keys())
    for v in varlist:
        if v.startswith("vtu") or v.startswith("vtl") or (v == 'b') or (v == "b_roll"):
            eib[v].attrs["units"] = "m"
    return eib

In [13]:
def load_WCSI_tracks():
    # Find available files
    WCSI_files = glob("RA_tracks/TRACK_WCSI-*.pkl") 
    # Load them
    tracks_with_WCSI = {}
    for f in WCSI_files:
        name = "TRACK-"+f.split("/")[-1].split('.')[0].split("-")[-1]
        with open(f, "rb") as file:
            tracks_with_WCSI[name] = pkl.load(file)
    return tracks_with_WCSI

In [14]:
def add_WCSI_flag(eib):

    tracks_WCSI = load_WCSI_tracks()
        
    L = []
    for source in tqdm(tracks_WCSI):
        # Transform the sources into dataframes
        t_WCSI_df = tracks_WCSI[source].to_dataframe()
        teib_df = eib.sel(source = source).squeeze().reset_coords().to_dataframe()
        for var in ["lon", "lat"]: # Round up the coordinates for the matching to work
            N = 0
            t_WCSI_df[var] = np.round(t_WCSI_df[var], N)
            teib_df[var] = np.round(teib_df[var], N)
        # Merge RABTracks and WCSI source on lon, lat and time
        t_WCSI_df = t_WCSI_df[~t_WCSI_df[["lon", "lat", "time",]].duplicated()] # Remove duplicates
        M = teib_df[["lon", "lat", "time",]].reset_index().merge(t_WCSI_df[["lon", "lat", "time", "WCSI"]], how = "left", on = ["lon", "lat", "time"])
        assert len(M) == len(teib_df) # Check that no duplicate were created
        M_xr = M.set_index("record").to_xarray().assign(source = source).set_coords("source")[["WCSI"]] # convert back to source and format
        L.append(M_xr)
    WCSI = xr.concat(L, dim = "source") # Merge the WCSI variables over sources
    eib = eib.merge(WCSI) # Merge back into the full RABTracks source
    eib = eib.assign(WCSI = eib.WCSI.fillna(False).astype(int)) # Convert to boolean
    
    return eib

In [15]:
def add_is_tc_is_etc_flags(eib):
    ## SyCLoPS: If flagged by SyCLoPS already
    ## IBTrACS: If nature is TS
    eib.is_tc.loc[dict(source="IBTrACS")] = (eib.nature == "TS")
    
    ## TRACK with CPS: 
    # If WCSI, then is_tc
    eib.is_tc.loc[dict(source="TRACK-ERA5")] = (eib.WCSI.loc[dict(source="TRACK-ERA5")])
    eib.is_tc.loc[dict(source="TRACK-JRA3Q")] = (eib.WCSI.loc[dict(source="TRACK-JRA3Q")])
    # Last WCS point after WCSI are also considered TCs
    eib["WCS"] = (np.abs(eib.b_roll) <= 15) & (eib.vtl_roll > 0)
    eib_df = eib.to_dataframe().reset_index()
    # Identify last WCSI point
    last_WCSI = eib_df[eib_df.is_tc.astype(bool)].sort_values("time").groupby(["source", "track_id"]).last()[["record", "time"]].reset_index()
    # Subset points post WCSI
    tmp = eib_df.merge(last_WCSI, on = ["track_id", "source"], suffixes=['', '_onset'])
    tmp = tmp.assign(delta = tmp.time - tmp.time_onset)
    post_WCSI = tmp[tmp.delta >= np.timedelta64(0)]
    # Identify first non-WCS point
    nonWCS = post_WCSI[~post_WCSI.WCS]
    first_nonWCS = nonWCS.sort_values("time").groupby(["source", "track_id"]).first()[["record", "time"]].reset_index()
    # List of WCS point just after WCSI
    post_WCSI = post_WCSI.merge(first_nonWCS, on = ["track_id", "source"], suffixes=['', '_nonWCS'])
    WCS_post_WCSI = post_WCSI[(post_WCSI.time - post_WCSI.time_nonWCS) < np.timedelta64(0)]
    eib.is_tc.loc[dict(record = WCS_post_WCSI.record.to_xarray(), source = WCS_post_WCSI.source.to_xarray())] = True

    # Is ETC if Cold Core Asymetric (CCA, TRACK with CPS) or EX (SyCLoPS) or ET nature (IBTrACS)
    eib["CCA"] = (np.abs(eib.b_roll) > 15) & (eib.vtl_roll < 0)
    eib["is_etc"] = (eib.CCA) | (eib.label == b'EX')
    eib.is_etc.loc[dict(source="IBTrACS")] = (eib.nature == "ET")

    return eib

In [16]:
def add_et_flag(eib):
    # Create ET variable
    eib["ET"]  = xr.DataArray(
        np.full(eib['lon'].shape, np.nan, dtype=object),
        dims=eib.dims,
        coords=eib.coords
    )

    # Convert to dataframe to be able to group by two variables
    eib_df = eib.to_dataframe().reset_index()

    # Identify last TS point
    TS = eib_df[eib_df.is_tc.fillna(False).astype(bool)]
    last_TS = TS.sort_values("time").groupby(["source", "track_id"]).last()[["record", "time"]].reset_index()
    eib.ET.loc[dict(record = last_TS.record.to_xarray(), source = last_TS.source.to_xarray())] = +1

    # Subset points post ET onset
    tmp = eib_df.merge(last_TS, on = ["track_id", "source"], suffixes=['', '_onset'])
    tmp = tmp.assign(delta = tmp.time - tmp.time_onset)
    post_onset = tmp[tmp.delta >= np.timedelta64(0)]

    # Identify first ETC point
    EX = post_onset[post_onset.is_etc.fillna(False).astype(bool)]
    first_EX = EX.sort_values("time").groupby(["source", "track_id"]).first()[["record", "time"]].reset_index()
    eib.ET.loc[dict(record = first_EX.record.to_xarray(), source = first_EX.source.to_xarray())] = -1

    # Convert ET to int8
    eib = eib.assign(ET = eib.ET.fillna(0).astype(np.int8))

    return eib

In [17]:
def save_old(eib, filename):
    print("... Saving ...")
    ## Specify time encoding
    eib.time.encoding['units'] = 'seconds since 1900-01-01'
    eib.time.encoding['calendar'] = 'standard'
    ## Changes types for lighter files
    ### Convert to float32
    varlist = list(eib.variables.keys())
    floatvars = list(set(varlist) & (set(["b_roll", "vtu_roll", "vtl_roll", "azimuth", "translation_speed", "lon", "lat"])))
    for var in floatvars:
        eib[var] = eib[var].astype(np.float32)
    ## Actual saving
    eib.to_netcdf(filename)


In [18]:
def save(eib, filename):
    print("... Saving ...")
    ## Specify time encoding
    eib.time.encoding['units'] = 'seconds since 1900-01-01'
    eib.time.encoding['calendar'] = 'standard'
    ## Changes types for lighter files
    ### Convert to float32
    varlist = list(eib.variables.keys())
    floatvars = list(set(varlist) & (set(["b_roll", "vtu_roll", "vtl_roll", "azimuth", "translation_speed", "lon", "lat"])))
    for var in floatvars:
        eib[var] = eib[var].astype(np.float32)
    ## Actual saving
    eib.to_netcdf(filename)

#Nested dictionary with variable names as keys and dictionaries of variable specific encodings as values, e.g., {"my_variable": {"dtype": "int16", "scale_factor": 0.1, "zlib": True}, ...}
#The h5netcdf engine supports both the NetCDF4-style compression encoding parameters {"zlib": True, "complevel": 9} and the h5py ones {"compression": "gzip", "compression_opts": 9}. This allows using any compression plugin installed in the HDF5 library, e.g. LZF.

In [19]:
def compress_file_chunked(input_path, output_path=None, chunk_size=1024 * 1024):
    if output_path is None:
        output_path = input_path + ".xz"

    with open(input_path, "rb") as f_in:
        with lzma.open(output_path, "wb") as f_out:
            while chunk := f_in.read(chunk_size):
                f_out.write(chunk)

    print(f"Compressed '{input_path}' -> '{output_path}'")

In [20]:
def produce_rabtracks(b, filename):
    print("============", b, "============")
    print("... Loading IBTrACS ...")
    ib = load_ibtracs(b)
    print("... Loading tracks ...")
    tracks = load_tracks()
    print("... Matching tracks ...")
    M, tracks_combined = matching(tracks, ib)
    print("... Gathering data ...")
    eib = gathering(tracks_combined, M, ib)
    print("... Retrieving IBTrACS attributes ...")
    eib = retrieve_ib_attrs(eib, ib)
    print("... Adding translation speed, azimuth and RMW...")
    eib = add_translation_speed_azimuth(eib)
    eib = add_radius_of_maximum_wind(eib)
    print("... Adding CPS parameters ...")
    eib = add_CPS_params(eib)
    print("... Adding WCSI flag ...")
    eib = add_WCSI_flag(eib)
    print("... Adding TC and ETC flags ...")
    eib = add_is_tc_is_etc_flags(eib)
    print("... Adding ET flag ...")
    eib = add_et_flag(eib)
    print("... Saving RABTracks ...")
    save(eib, filename)
    print("... Compression ...")
    compress_file_chunked(filename)
    del ib, tracks, eib

for b in BASINS:
    filename = "RABTracks/RABTracks_"+b+".nc"
    if not os.path.isfile(filename):
        eib = produce_rabtracks(b, filename)

============ WP ============
... Loading IBTrACS ...
... Loading tracks ...


100%|██████████| 8/8 [00:12<00:00,  1.62s/it]


... Matching tracks ...


 88%|████████▊ | 7/8 [01:00<00:07,  7.13s/it]/home/users/sbourdin/.conda/envs/huracanpy/lib/python3.13/site-packages/haversine/haversine.py:131: RuntimeWarning: invalid value encountered in arcsin
  return 2 * asin(sqrt(d))
100%|██████████| 8/8 [01:08<00:00,  8.60s/it]


... Gathering data ...


...

... Adding CPS parameters ...


100%|██████████| 2/2 [00:02<00:00,  1.11s/it]


... Adding WCSI flag ...


100%|██████████| 2/2 [00:01<00:00,  1.20it/s]


... Adding TC and ETC flags ...
... Adding ET flag ...
... Saving RABTracks ...
... Saving ...
... Compression ...
Compressed 'RABTracks/RABTracks_WP.nc' -> 'RABTracks/RABTracks_WP.nc.xz'


In [21]:
ds = xr.open_dataset("RABTracks/RABTracks_WP.nc")

In [22]:
descr=ds.to_dataframe().describe()

In [23]:
descr

,lon,lat,pres,wind10,vo850,lon_pres,lat_pres,lon_wind,lat_wind,is_tc,...,radius_max_wind,radius_max_wind_roll,vtu,vtl,b,vtu_roll,vtl_roll,b_roll,WCSI,ET
count,926801.000000,926801.000000,806915.000000,806914.000000,488420.000000,475221.000000,475221.000000,525683.000000,525683.000000,1.003877e+06,...,475220.000000,399448.000000,299774.000000,299774.000000,299794.000000,279682.000000,279682.000000,279702.000000,2.167938e+06,2.167938e+06
mean,111.725021,21.020308,999.270386,14.725480,10.639352,140.149881,20.141671,141.712435,20.727758,4.240500e-01,...,342.887240,312.553096,-7.161695,19.718674,5.818801,-3.798171,22.098917,5.762619,3.019505e-02,4.034248e-03
min,-180.000000,-15.854700,900.525879,1.633506,-2.929491,0.535486,-11.840790,0.000000,-20.500000,0.000000e+00,...,0.341689,10.136772,-651.391174,-648.333618,-126.263901,-603.587463,-581.105164,-58.945751,0.000000e+00,-1.000000e+00
25%,116.350861,11.881229,994.954468,9.876287,4.896891,121.691900,10.946770,121.950000,12.020800,0.000000e+00,...,137.982093,150.953144,-17.843491,-2.108867,-2.606689,-14.247988,0.330354,-2.014795,0.000000e+00,0.000000e+00
50%,132.419052,18.000000,1003.174988,13.054399,8.358456,136.944500,17.313180,138.375000,18.000000,0.000000e+00,...,293.814391,284.477202,2.428208,18.217140,1.530029,3.066995,19.263464,1.503955,0.000000e+00,0.000000e+00
75%,147.677536,26.318890,1007.416016,17.875673,14.502632,153.959900,25.497210,156.415000,25.860300,1.000000e+00,...,522.900801,450.566973,30.298532,51.277451,7.250977,32.495049,51.973415,6.704113,0.000000e+00,0.000000e+00
max,179.999084,89.500000,1035.470947,60.191479,52.647949,359.838100,77.377190,357.500000,79.229800,1.000000e+00,...,4900.392918,3617.344426,417.704315,717.064270,182.597397,367.921600,534.951172,162.530197,1.000000e+00,1.000000e+00
std,82.014847,13.003857,12.250398,6.807190,7.629264,27.825575,12.824314,28.618100,12.388809,4.941982e-01,...,233.862117,184.546605,99.369400,78.895569,18.552107,95.220947,72.764824,17.055220,1.711237e-01,9.220858e-02
